In [0]:
dbutils.widgets.removeAll()

In [0]:
# Celda 2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType
from pyspark.sql.functions import current_timestamp, col

In [0]:
# Celda 3
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlaiza082026")

In [0]:
# Celda 4
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/goodreads_bbe/books_1_Best_Books_Ever.csv"

In [0]:
# Celda 5
goodreads_schema = StructType(fields=[
    StructField("title", StringType(), True),
    StructField("series", StringType(), True),
    StructField("author", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("description", StringType(), True),
    StructField("language", StringType(), True),
    StructField("isbn", StringType(), True),
    StructField("genres", StringType(), True),
    StructField("characters", StringType(), True),
    StructField("bookFormat", StringType(), True),
    StructField("edition", StringType(), True),
    StructField("pages", IntegerType(), True),
    StructField("publisher", StringType(), True),
    StructField("publishDate", StringType(), True),
    StructField("firstPublishDate", StringType(), True),
    StructField("awards", StringType(), True),
    StructField("numRatings", LongType(), True),
    StructField("ratingsByStars", StringType(), True),
    StructField("likedPercent", DoubleType(), True),
    StructField("setting", StringType(), True),
    StructField("bbeScore", LongType(), True),
    StructField("bbeVotes", LongType(), True),
    StructField("price", DoubleType(), True)
])

In [0]:
# Ver qué hay en el contenedor raw
display(dbutils.fs.ls(f"abfss://raw@{storageName}.dfs.core.windows.net/"))

In [0]:
display(dbutils.fs.ls(f"abfss://raw@{storageName}.dfs.core.windows.net/goodreads_bbe/"))

In [0]:
display(dbutils.fs.ls(f"abfss://raw@{storageName}.dfs.core.windows.net/amazon_bestsellers/"))

In [0]:
# Celda 6
df_goodreads = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .schema(goodreads_schema) \
    .csv(ruta)
    

In [0]:
# Celda 7
goodreads_final_df = df_goodreads.withColumn("ingestion_date", current_timestamp())

In [0]:
# Celda 8
goodreads_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.goodreads_books")